In [1]:
# Cell 1: Imports, device, constants, and run flags

import os
import copy
import glob
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.io import wavfile
from scipy.signal import spectrogram, resample
from scipy.fftpack import dct
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder

# Device + shared constants
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
NUM_CLASSES = 10
INPUT_SHAPE = (3, 180, 180)
BATCH_SIZE = 32

LEARNING_RATE_FC = 1e-3
LEARNING_RATE_CNN = 1e-4
LEARNING_RATE_RMSPROP = 1e-4
DEBUG_EPOCHS = 1

# Overnight control flags
RUN_SMOKE_TESTS = True
RUN_DEBUG = False
RUN_FULL_IMAGE_MODELS = True
RUN_FULL_AUDIO_MODELS = True
RUN_GAN = True

# Paths
IMAGE_DIR = "Data/images_original"
AUDIO_DIR = "Data/genres_original"
CHECKPOINT_DIR = "checkpoints"
IMAGE_RESULTS_CSV = "results_image_models.csv"
ALL_RESULTS_CSV = "results_all_models.csv"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Using device:", DEVICE)
print("SEED:", SEED)
print("RUN_SMOKE_TESTS:", RUN_SMOKE_TESTS)
print("RUN_DEBUG:", RUN_DEBUG)
print("RUN_FULL_IMAGE_MODELS:", RUN_FULL_IMAGE_MODELS)
print("RUN_FULL_AUDIO_MODELS:", RUN_FULL_AUDIO_MODELS)
print("RUN_GAN:", RUN_GAN)


Using device: cpu


In [2]:
# Cell 2: Image dataset, fixed transform, 70/20/10 split, and sanity checks

image_transform = transforms.Compose([
    transforms.Resize((180, 180)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

image_dataset = ImageFolder(root=IMAGE_DIR, transform=image_transform)
print("Image classes:", image_dataset.classes)
print("Total image samples:", len(image_dataset))

total_size = len(image_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.2 * total_size)
test_size = total_size - train_size - val_size

split_generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset, test_dataset = random_split(
    image_dataset,
    [train_size, val_size, test_size],
    generator=split_generator,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Image split sizes (70/20/10): train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

def split_class_counts(subset, base_dataset):
    counts = {cls: 0 for cls in base_dataset.classes}
    for idx in subset.indices:
        class_idx = base_dataset.targets[idx]
        counts[base_dataset.classes[class_idx]] += 1
    return counts

print("Image train class counts:", split_class_counts(train_dataset, image_dataset))
print("Image val class counts:", split_class_counts(val_dataset, image_dataset))
print("Image test class counts:", split_class_counts(test_dataset, image_dataset))

images, labels = next(iter(train_loader))
print("Image batch shape:", tuple(images.shape))
print("Label batch shape:", tuple(labels.shape))
print("Image value range after transform:", float(images.min()), float(images.max()))


Classes: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Class to index: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}
Total image samples: 999
Train size: 699
Validation size: 199
Test size: 101
Image batch shape: torch.Size([32, 3, 180, 180])
Label batch shape: torch.Size([32])
Example labels: tensor([4, 9, 4, 5, 6, 4, 5, 2, 1, 9])


In [3]:
# Cell 3: Utility functions

def set_seed(seed=SEED):
    """Set random seeds for reproducibility across Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def count_parameters(model):
    """Return number of trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def smoke_test_model(model, input_shape=INPUT_SHAPE, batch_size=4, num_classes=NUM_CLASSES, device=DEVICE):
    """
    Quick shape/device sanity check for a model.
    Ensures output shape is [batch_size, num_classes].
    """
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        x = torch.randn(batch_size, *input_shape).to(device)
        y = model(x)

    assert y.shape == (batch_size, num_classes), (
        f"Smoke test failed: expected {(batch_size, num_classes)}, got {tuple(y.shape)}"
    )
    print(f"Smoke test passed. Output shape: {tuple(y.shape)}")
    print(f"Trainable parameters: {count_parameters(model):,}")


In [4]:
# Cell 4: Shared training/evaluation + robust result/checkpoint saving

criterion = nn.CrossEntropyLoss()
results_records = []
RESULT_COLUMNS = [
    "model_name", "input_type", "architecture", "optimizer", "epochs_run",
    "best_val_acc", "best_epoch", "test_loss", "test_acc"
]


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_one_epoch(model, loader, optimizer, criterion, device=DEVICE):
    model.train()
    running_loss, total = 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        total += images.size(0)
    return running_loss / total


def evaluate(model, loader, criterion, device=DEVICE):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, 100.0 * correct / total


def train_model(model, model_name, architecture, optimizer_name, epochs, train_loader, val_loader, criterion, lr, checkpoint_path, device=DEVICE):
    set_seed(SEED)
    model = model.to(device)

    if optimizer_name.lower() == "rmsprop":
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    elif optimizer_name.lower() == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"[{model_name}] Epoch {epoch}/{epochs} | Train Loss {train_loss:.4f} | Val Loss {val_loss:.4f} | Val Acc {val_acc:.2f}%")

    model.load_state_dict(best_state)
    torch.save(best_state, checkpoint_path)
    print(f"Saved best checkpoint to {checkpoint_path}")
    return model, history


def test_model(model, loader, criterion, device=DEVICE):
    return evaluate(model, loader, criterion, device)


def append_result(model_name, input_type, architecture, optimizer_name, epochs_run, history, test_loss, test_acc):
    best_epoch_idx = int(np.argmax(history["val_acc"]))
    results_records.append({
        "model_name": model_name,
        "input_type": input_type,
        "architecture": architecture,
        "optimizer": optimizer_name,
        "epochs_run": int(epochs_run),
        "best_val_acc": float(history["val_acc"][best_epoch_idx]),
        "best_epoch": int(best_epoch_idx + 1),
        "test_loss": float(test_loss),
        "test_acc": float(test_acc),
    })


def save_results_csv(path):
    df = pd.DataFrame(results_records)
    if df.empty:
        df = pd.DataFrame(columns=RESULT_COLUMNS)
    else:
        df = df[RESULT_COLUMNS].sort_values(["model_name", "epochs_run"]).reset_index(drop=True)
    df.to_csv(path, index=False)
    print(f"Saved {len(df)} rows to {path}")
    return df


In [5]:
# Cell 5: Net1 model definition only

class Net1(nn.Module):
    """Net1: fully connected network with exactly two hidden layers."""
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        # Net1 has many parameters because a 180x180 RGB image is flattened directly.
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3 * 180 * 180, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x


In [6]:
# Cell 6: Net1 smoke test and optional 1-epoch debug run

set_seed(SEED)
net1 = Net1()
if RUN_SMOKE_TESTS:
    smoke_test_model(net1)

if RUN_DEBUG:
    set_seed(SEED)
    net1_debug = Net1()
    net1_debug, net1_debug_history = train_model(
        model=net1_debug,
        model_name="Net1",
        architecture="FC(Flatten->512->128->10)",
        optimizer_name="Adam",
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        lr=LEARNING_RATE_FC,
        checkpoint_path=os.path.join(CHECKPOINT_DIR, "net1_debug_best.pt"),
    )
    dbg_test_loss, dbg_test_acc = test_model(net1_debug, test_loader, criterion)
    print(f"Net1 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net1 debug run skipped (RUN_DEBUG=False).")


Smoke test passed. Output shape: (4, 10)
Trainable parameters: 49,833,866
[Net1] Epoch 1/1 | Train Loss: 7.2287 | Val Loss: 2.5014 | Val Acc: 13.57%
Net1 debug test loss: 2.6223, test acc: 12.87%


In [18]:
# Cell 7: Net1 50-epoch and 100-epoch training calls

if RUN_FULL_IMAGE_MODELS:
    print("\n===== Training Net1 (image input) =====")
    for n_epochs in [50, 100]:
        print(f"\nStarting Net1 for {n_epochs} epochs...")
        set_seed(SEED)
        net1_run = Net1()
        net1_run, net1_history = train_model(
            model=net1_run,
            model_name="Net1",
            architecture="FC(Flatten->512->128->10)",
            optimizer_name="Adam",
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            lr=LEARNING_RATE_FC,
            checkpoint_path=os.path.join(CHECKPOINT_DIR, f"net1_{n_epochs}_best.pt"),
        )
        net1_test_loss, net1_test_acc = test_model(net1_run, test_loader, criterion)
        append_result("Net1", "image_mel_180x180", "FC(Flatten->512->128->10)", "Adam", n_epochs, net1_history, net1_test_loss, net1_test_acc)
        save_results_csv(IMAGE_RESULTS_CSV)
        save_results_csv(ALL_RESULTS_CSV)
        print(f"Net1 ({n_epochs} epochs) test acc: {net1_test_acc:.2f}%")
else:
    print("Net1 full training skipped (RUN_FULL_IMAGE_MODELS=False).")


[Net1] Epoch 1/50 | Train Loss: 7.2287 | Val Loss: 2.5014 | Val Acc: 13.57%
[Net1] Epoch 2/50 | Train Loss: 2.3058 | Val Loss: 2.4359 | Val Acc: 18.09%
[Net1] Epoch 3/50 | Train Loss: 2.2083 | Val Loss: 2.1628 | Val Acc: 25.13%
[Net1] Epoch 4/50 | Train Loss: 2.0729 | Val Loss: 1.9184 | Val Acc: 32.16%
[Net1] Epoch 5/50 | Train Loss: 1.9228 | Val Loss: 1.9470 | Val Acc: 27.14%
[Net1] Epoch 6/50 | Train Loss: 1.7945 | Val Loss: 1.8311 | Val Acc: 33.17%
[Net1] Epoch 7/50 | Train Loss: 1.7453 | Val Loss: 1.9864 | Val Acc: 25.13%
[Net1] Epoch 8/50 | Train Loss: 1.6656 | Val Loss: 1.9724 | Val Acc: 29.65%
[Net1] Epoch 9/50 | Train Loss: 1.6922 | Val Loss: 1.9179 | Val Acc: 33.17%
[Net1] Epoch 10/50 | Train Loss: 1.6634 | Val Loss: 2.0149 | Val Acc: 30.65%
[Net1] Epoch 11/50 | Train Loss: 1.6721 | Val Loss: 1.8285 | Val Acc: 34.17%
[Net1] Epoch 12/50 | Train Loss: 1.4640 | Val Loss: 1.9323 | Val Acc: 30.15%
[Net1] Epoch 13/50 | Train Loss: 1.4018 | Val Loss: 1.7550 | Val Acc: 36.18%
[Net1] E

[Net1] Epoch 57/100 | Train Loss: 0.0604 | Val Loss: 2.3396 | Val Acc: 44.22%
[Net1] Epoch 58/100 | Train Loss: 0.0840 | Val Loss: 2.5117 | Val Acc: 44.72%
[Net1] Epoch 59/100 | Train Loss: 0.0426 | Val Loss: 2.2494 | Val Acc: 44.72%
[Net1] Epoch 60/100 | Train Loss: 0.0454 | Val Loss: 2.2864 | Val Acc: 47.24%
[Net1] Epoch 61/100 | Train Loss: 0.0854 | Val Loss: 2.3437 | Val Acc: 43.22%
[Net1] Epoch 62/100 | Train Loss: 0.2100 | Val Loss: 2.3035 | Val Acc: 44.22%
[Net1] Epoch 63/100 | Train Loss: 0.1097 | Val Loss: 2.1970 | Val Acc: 45.23%
[Net1] Epoch 64/100 | Train Loss: 0.3322 | Val Loss: 2.1755 | Val Acc: 41.71%
[Net1] Epoch 65/100 | Train Loss: 0.2536 | Val Loss: 2.4666 | Val Acc: 44.22%
[Net1] Epoch 66/100 | Train Loss: 0.2365 | Val Loss: 3.3261 | Val Acc: 40.70%
[Net1] Epoch 67/100 | Train Loss: 0.5174 | Val Loss: 2.9382 | Val Acc: 37.69%
[Net1] Epoch 68/100 | Train Loss: 0.3063 | Val Loss: 2.2335 | Val Acc: 45.23%
[Net1] Epoch 69/100 | Train Loss: 0.2595 | Val Loss: 2.2475 | Va

In [8]:
# Cell 8: Net2 model definition only

class Net2(nn.Module):
    """
    Net2: Figure-1-style CNN with AdaptiveAvgPool2d((2,2)).
    Flattened feature size before classifier: 64 * 2 * 2 = 256.
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.AdaptiveAvgPool2d((2, 2)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 2 * 2, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [9]:
# Cell 9: Net2 smoke test and optional 1-epoch debug run

set_seed(SEED)
net2 = Net2()
if RUN_SMOKE_TESTS:
    smoke_test_model(net2)

if RUN_DEBUG:
    set_seed(SEED)
    net2_debug = Net2()
    net2_debug, net2_debug_history = train_model(
        model=net2_debug,
        model_name="Net2",
        architecture="CNN(Fig1+AdaptivePool 4conv+2pool+fc256)",
        optimizer_name="Adam",
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        lr=LEARNING_RATE_CNN,
        checkpoint_path=os.path.join(CHECKPOINT_DIR, "net2_debug_best.pt"),
    )
    dbg_test_loss, dbg_test_acc = test_model(net2_debug, test_loader, criterion)
    print(f"Net2 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net2 debug run skipped (RUN_DEBUG=False).")


Smoke test passed. Output shape: (4, 10)
Trainable parameters: 33,245,994
[Net2] Epoch 1/1 | Train Loss: 2.3134 | Val Loss: 2.3035 | Val Acc: 9.55%
Net2 debug test loss: 2.2953, test acc: 11.88%


In [20]:
# Cell 10: Net2 50-epoch and 100-epoch training calls

if RUN_FULL_IMAGE_MODELS:
    print("\n===== Training Net2 (image input) =====")
    for n_epochs in [50, 100]:
        print(f"\nStarting Net2 for {n_epochs} epochs...")
        set_seed(SEED)
        net2_run = Net2()
        net2_run, net2_history = train_model(
            model=net2_run,
            model_name="Net2",
            architecture="CNN(Fig1+AdaptivePool 4conv+2pool+fc256)",
            optimizer_name="Adam",
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            lr=LEARNING_RATE_CNN,
            checkpoint_path=os.path.join(CHECKPOINT_DIR, f"net2_{n_epochs}_best.pt"),
        )
        net2_test_loss, net2_test_acc = test_model(net2_run, test_loader, criterion)
        append_result("Net2", "image_mel_180x180", "CNN(Fig1+AdaptivePool 4conv+2pool+fc256)", "Adam", n_epochs, net2_history, net2_test_loss, net2_test_acc)
        save_results_csv(IMAGE_RESULTS_CSV)
        save_results_csv(ALL_RESULTS_CSV)
        print(f"Net2 ({n_epochs} epochs) test acc: {net2_test_acc:.2f}%")
else:
    print("Net2 full training skipped (RUN_FULL_IMAGE_MODELS=False).")


[Net2] Epoch 1/50 | Train Loss: 2.3134 | Val Loss: 2.3035 | Val Acc: 9.55%
[Net2] Epoch 2/50 | Train Loss: 2.2862 | Val Loss: 2.2640 | Val Acc: 15.08%
[Net2] Epoch 3/50 | Train Loss: 2.1933 | Val Loss: 2.1219 | Val Acc: 20.60%
[Net2] Epoch 4/50 | Train Loss: 2.0372 | Val Loss: 1.9948 | Val Acc: 24.62%
[Net2] Epoch 5/50 | Train Loss: 1.9642 | Val Loss: 1.9617 | Val Acc: 25.13%
[Net2] Epoch 6/50 | Train Loss: 1.8785 | Val Loss: 1.9326 | Val Acc: 26.63%
[Net2] Epoch 7/50 | Train Loss: 1.7966 | Val Loss: 1.9165 | Val Acc: 28.14%
[Net2] Epoch 8/50 | Train Loss: 1.7308 | Val Loss: 1.8451 | Val Acc: 30.65%
[Net2] Epoch 9/50 | Train Loss: 1.6857 | Val Loss: 1.7515 | Val Acc: 40.20%
[Net2] Epoch 10/50 | Train Loss: 1.5866 | Val Loss: 1.7351 | Val Acc: 41.71%
[Net2] Epoch 11/50 | Train Loss: 1.4813 | Val Loss: 1.7206 | Val Acc: 37.69%
[Net2] Epoch 12/50 | Train Loss: 1.3803 | Val Loss: 1.6899 | Val Acc: 41.21%
[Net2] Epoch 13/50 | Train Loss: 1.3026 | Val Loss: 1.7112 | Val Acc: 41.21%
[Net2] Ep

[Net2] Epoch 57/100 | Train Loss: 0.0195 | Val Loss: 2.6386 | Val Acc: 51.76%
[Net2] Epoch 58/100 | Train Loss: 0.0175 | Val Loss: 2.7066 | Val Acc: 51.76%
[Net2] Epoch 59/100 | Train Loss: 0.0336 | Val Loss: 2.6667 | Val Acc: 51.76%
[Net2] Epoch 60/100 | Train Loss: 0.0123 | Val Loss: 2.6852 | Val Acc: 51.26%
[Net2] Epoch 61/100 | Train Loss: 0.0252 | Val Loss: 2.7299 | Val Acc: 50.25%
[Net2] Epoch 62/100 | Train Loss: 0.0195 | Val Loss: 2.8283 | Val Acc: 51.26%
[Net2] Epoch 63/100 | Train Loss: 0.0322 | Val Loss: 2.8897 | Val Acc: 51.26%
[Net2] Epoch 64/100 | Train Loss: 0.0178 | Val Loss: 2.7777 | Val Acc: 51.26%
[Net2] Epoch 65/100 | Train Loss: 0.0088 | Val Loss: 2.8165 | Val Acc: 51.26%
[Net2] Epoch 66/100 | Train Loss: 0.0254 | Val Loss: 2.8343 | Val Acc: 52.76%
[Net2] Epoch 67/100 | Train Loss: 0.0091 | Val Loss: 2.8009 | Val Acc: 50.75%
[Net2] Epoch 68/100 | Train Loss: 0.0073 | Val Loss: 2.7882 | Val Acc: 52.76%
[Net2] Epoch 69/100 | Train Loss: 0.0118 | Val Loss: 2.8180 | Va

In [11]:
# Cell 11: Net3 model definition only

class Net3(nn.Module):
    """
    Net3: Same CNN as Net2 + BatchNorm2d, with AdaptiveAvgPool2d((2,2)).
    Flattened feature size before classifier: 64 * 2 * 2 = 256.
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.AdaptiveAvgPool2d((2, 2)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 2 * 2, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [12]:
# Cell 12: Net3 smoke test and optional 1-epoch debug run

set_seed(SEED)
net3 = Net3()
if RUN_SMOKE_TESTS:
    smoke_test_model(net3)

if RUN_DEBUG:
    set_seed(SEED)
    net3_debug = Net3()
    net3_debug, net3_debug_history = train_model(
        model=net3_debug,
        model_name="Net3",
        architecture="CNN+BN(Fig1+AdaptivePool 4conv+2pool+fc256)",
        optimizer_name="Adam",
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        lr=LEARNING_RATE_CNN,
        checkpoint_path=os.path.join(CHECKPOINT_DIR, "net3_debug_best.pt"),
    )
    dbg_test_loss, dbg_test_acc = test_model(net3_debug, test_loader, criterion)
    print(f"Net3 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net3 debug run skipped (RUN_DEBUG=False).")


Smoke test passed. Output shape: (4, 10)
Trainable parameters: 33,246,378
[Net3] Epoch 1/1 | Train Loss: 3.6786 | Val Loss: 2.3955 | Val Acc: 11.06%
Net3 debug test loss: 2.4504, test acc: 5.94%


In [19]:
# Cell 13: Net3 50-epoch and 100-epoch training calls

if RUN_FULL_IMAGE_MODELS:
    print("\n===== Training Net3 (image input) =====")
    for n_epochs in [50, 100]:
        print(f"\nStarting Net3 for {n_epochs} epochs...")
        set_seed(SEED)
        net3_run = Net3()
        net3_run, net3_history = train_model(
            model=net3_run,
            model_name="Net3",
            architecture="CNN+BN(Fig1+AdaptivePool 4conv+2pool+fc256)",
            optimizer_name="Adam",
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            lr=LEARNING_RATE_CNN,
            checkpoint_path=os.path.join(CHECKPOINT_DIR, f"net3_{n_epochs}_best.pt"),
        )
        net3_test_loss, net3_test_acc = test_model(net3_run, test_loader, criterion)
        append_result("Net3", "image_mel_180x180", "CNN+BN(Fig1+AdaptivePool 4conv+2pool+fc256)", "Adam", n_epochs, net3_history, net3_test_loss, net3_test_acc)
        save_results_csv(IMAGE_RESULTS_CSV)
        save_results_csv(ALL_RESULTS_CSV)
        print(f"Net3 ({n_epochs} epochs) test acc: {net3_test_acc:.2f}%")
else:
    print("Net3 full training skipped (RUN_FULL_IMAGE_MODELS=False).")


[Net3] Epoch 1/50 | Train Loss: 3.6786 | Val Loss: 2.3955 | Val Acc: 11.06%
[Net3] Epoch 2/50 | Train Loss: 1.8573 | Val Loss: 1.8509 | Val Acc: 40.20%
[Net3] Epoch 3/50 | Train Loss: 1.4405 | Val Loss: 1.6980 | Val Acc: 39.70%
[Net3] Epoch 4/50 | Train Loss: 1.1335 | Val Loss: 1.3765 | Val Acc: 55.28%
[Net3] Epoch 5/50 | Train Loss: 0.8069 | Val Loss: 1.4024 | Val Acc: 51.26%
[Net3] Epoch 6/50 | Train Loss: 0.6208 | Val Loss: 1.4141 | Val Acc: 53.27%
[Net3] Epoch 7/50 | Train Loss: 0.3919 | Val Loss: 1.2710 | Val Acc: 54.77%
[Net3] Epoch 8/50 | Train Loss: 0.2375 | Val Loss: 1.3077 | Val Acc: 55.28%
[Net3] Epoch 9/50 | Train Loss: 0.1999 | Val Loss: 1.1879 | Val Acc: 60.80%
[Net3] Epoch 10/50 | Train Loss: 0.1335 | Val Loss: 1.1473 | Val Acc: 59.30%
[Net3] Epoch 11/50 | Train Loss: 0.0912 | Val Loss: 1.1612 | Val Acc: 61.81%
[Net3] Epoch 12/50 | Train Loss: 0.0725 | Val Loss: 1.1772 | Val Acc: 61.31%
[Net3] Epoch 13/50 | Train Loss: 0.0558 | Val Loss: 1.2221 | Val Acc: 58.29%
[Net3] E

[Net3] Epoch 57/100 | Train Loss: 0.0063 | Val Loss: 1.3542 | Val Acc: 61.31%
[Net3] Epoch 58/100 | Train Loss: 0.0124 | Val Loss: 1.2661 | Val Acc: 62.81%
[Net3] Epoch 59/100 | Train Loss: 0.0114 | Val Loss: 1.2592 | Val Acc: 61.81%
[Net3] Epoch 60/100 | Train Loss: 0.0092 | Val Loss: 1.2494 | Val Acc: 59.80%
[Net3] Epoch 61/100 | Train Loss: 0.0097 | Val Loss: 1.2014 | Val Acc: 61.81%
[Net3] Epoch 62/100 | Train Loss: 0.0045 | Val Loss: 1.2267 | Val Acc: 63.82%
[Net3] Epoch 63/100 | Train Loss: 0.0066 | Val Loss: 1.3641 | Val Acc: 61.31%
[Net3] Epoch 64/100 | Train Loss: 0.0091 | Val Loss: 1.1917 | Val Acc: 64.82%
[Net3] Epoch 65/100 | Train Loss: 0.0116 | Val Loss: 1.6463 | Val Acc: 56.78%
[Net3] Epoch 66/100 | Train Loss: 0.0181 | Val Loss: 1.5240 | Val Acc: 63.82%
[Net3] Epoch 67/100 | Train Loss: 0.0086 | Val Loss: 1.2454 | Val Acc: 62.81%
[Net3] Epoch 68/100 | Train Loss: 0.0087 | Val Loss: 1.4861 | Val Acc: 59.80%
[Net3] Epoch 69/100 | Train Loss: 0.0128 | Val Loss: 1.7858 | Va

In [14]:
# Cell 14: Net4 setup using Net3 architecture and RMSprop

Net4 = Net3
NET4_ARCH = "CNN+BN(Fig1+AdaptivePool 4conv+2pool+fc256)"
NET4_OPTIMIZER = "RMSprop"

print("Net4 setup complete: Net3 architecture + RMSprop optimizer.")


Net4 setup complete: using Net3 architecture with RMSprop optimizer.


In [15]:
# Cell 15: Net4 smoke test and optional 1-epoch debug run

set_seed(SEED)
net4 = Net4()
if RUN_SMOKE_TESTS:
    smoke_test_model(net4)

if RUN_DEBUG:
    set_seed(SEED)
    net4_debug = Net4()
    net4_debug, net4_debug_history = train_model(
        model=net4_debug,
        model_name="Net4",
        architecture=NET4_ARCH,
        optimizer_name=NET4_OPTIMIZER,
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        lr=LEARNING_RATE_RMSPROP,
        checkpoint_path=os.path.join(CHECKPOINT_DIR, "net4_debug_best.pt"),
    )
    dbg_test_loss, dbg_test_acc = test_model(net4_debug, test_loader, criterion)
    print(f"Net4 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net4 debug run skipped (RUN_DEBUG=False).")


Smoke test passed. Output shape: (4, 10)
Trainable parameters: 33,246,378
[Net4] Epoch 1/1 | Train Loss: 20.4306 | Val Loss: 4.1397 | Val Acc: 14.57%
Net4 debug test loss: 4.2102, test acc: 13.86%


In [21]:
# Cell 16: Net4 50-epoch and 100-epoch training calls

if RUN_FULL_IMAGE_MODELS:
    print("\n===== Training Net4 (image input, Net3 architecture + RMSprop) =====")
    for n_epochs in [50, 100]:
        print(f"\nStarting Net4 for {n_epochs} epochs...")
        set_seed(SEED)
        net4_run = Net4()
        net4_run, net4_history = train_model(
            model=net4_run,
            model_name="Net4",
            architecture=NET4_ARCH,
            optimizer_name=NET4_OPTIMIZER,
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            lr=LEARNING_RATE_RMSPROP,
            checkpoint_path=os.path.join(CHECKPOINT_DIR, f"net4_{n_epochs}_best.pt"),
        )
        net4_test_loss, net4_test_acc = test_model(net4_run, test_loader, criterion)
        append_result("Net4", "image_mel_180x180", NET4_ARCH, NET4_OPTIMIZER, n_epochs, net4_history, net4_test_loss, net4_test_acc)
        save_results_csv(IMAGE_RESULTS_CSV)
        save_results_csv(ALL_RESULTS_CSV)
        print(f"Net4 ({n_epochs} epochs) test acc: {net4_test_acc:.2f}%")
else:
    print("Net4 full training skipped (RUN_FULL_IMAGE_MODELS=False).")


[Net4] Epoch 1/50 | Train Loss: 20.4306 | Val Loss: 4.1397 | Val Acc: 14.57%
[Net4] Epoch 2/50 | Train Loss: 2.6550 | Val Loss: 2.1108 | Val Acc: 26.63%
[Net4] Epoch 3/50 | Train Loss: 1.8362 | Val Loss: 1.8488 | Val Acc: 32.66%
[Net4] Epoch 4/50 | Train Loss: 1.8907 | Val Loss: 1.7971 | Val Acc: 40.20%
[Net4] Epoch 5/50 | Train Loss: 1.6463 | Val Loss: 1.7656 | Val Acc: 32.66%
[Net4] Epoch 6/50 | Train Loss: 1.5047 | Val Loss: 2.0616 | Val Acc: 42.71%
[Net4] Epoch 7/50 | Train Loss: 1.4616 | Val Loss: 1.6639 | Val Acc: 45.23%
[Net4] Epoch 8/50 | Train Loss: 1.1321 | Val Loss: 1.7070 | Val Acc: 46.23%
[Net4] Epoch 9/50 | Train Loss: 1.1690 | Val Loss: 1.9467 | Val Acc: 35.68%
[Net4] Epoch 10/50 | Train Loss: 1.2191 | Val Loss: 1.2976 | Val Acc: 57.79%
[Net4] Epoch 11/50 | Train Loss: 0.8075 | Val Loss: 1.6775 | Val Acc: 51.26%
[Net4] Epoch 12/50 | Train Loss: 0.7202 | Val Loss: 2.1103 | Val Acc: 39.20%
[Net4] Epoch 13/50 | Train Loss: 0.7568 | Val Loss: 1.3421 | Val Acc: 57.29%
[Net4] 

[Net4] Epoch 57/100 | Train Loss: 0.0188 | Val Loss: 1.5931 | Val Acc: 58.79%
[Net4] Epoch 58/100 | Train Loss: 0.0208 | Val Loss: 1.4896 | Val Acc: 58.79%
[Net4] Epoch 59/100 | Train Loss: 0.0156 | Val Loss: 1.5040 | Val Acc: 59.30%
[Net4] Epoch 60/100 | Train Loss: 0.0121 | Val Loss: 2.2078 | Val Acc: 51.26%
[Net4] Epoch 61/100 | Train Loss: 0.0214 | Val Loss: 1.5067 | Val Acc: 60.30%
[Net4] Epoch 62/100 | Train Loss: 0.0213 | Val Loss: 1.4955 | Val Acc: 59.80%
[Net4] Epoch 63/100 | Train Loss: 0.0156 | Val Loss: 1.4789 | Val Acc: 59.30%
[Net4] Epoch 64/100 | Train Loss: 0.0082 | Val Loss: 1.6884 | Val Acc: 58.29%
[Net4] Epoch 65/100 | Train Loss: 0.0270 | Val Loss: 2.7593 | Val Acc: 45.23%
[Net4] Epoch 66/100 | Train Loss: 1.0883 | Val Loss: 1.5806 | Val Acc: 58.79%
[Net4] Epoch 67/100 | Train Loss: 0.0137 | Val Loss: 1.5400 | Val Acc: 57.79%
[Net4] Epoch 68/100 | Train Loss: 0.0123 | Val Loss: 1.5517 | Val Acc: 59.80%
[Net4] Epoch 69/100 | Train Loss: 0.0147 | Val Loss: 1.6018 | Va

In [22]:
# Cell 17: Results table and CSV saving for Net1-Net4

image_results_df = pd.DataFrame([r for r in results_records if r.get("model_name") in {"Net1", "Net2", "Net3", "Net4"}])
if image_results_df.empty:
    image_results_df = pd.DataFrame(columns=RESULT_COLUMNS)
    print("No image-model runs are logged yet.")
else:
    image_results_df = image_results_df[RESULT_COLUMNS].sort_values(["model_name", "epochs_run"]).reset_index(drop=True)
    display(image_results_df)

image_results_df.to_csv(IMAGE_RESULTS_CSV, index=False)
print(f"Saved image results to {IMAGE_RESULTS_CSV}")


,model_name,architecture,optimizer,epochs,final_training_loss,final_validation_loss,final_validation_accuracy,best_validation_accuracy,best_epoch,test_loss,test_accuracy
0,Net1,FC(Flatten->512->128->10),Adam,50,0.304235,2.298603,48.743719,48.743719,50,1.882553,48.514851
1,Net1,FC(Flatten->512->128->10),Adam,100,0.074583,3.479503,37.688442,51.256281,80,1.913278,51.485149
2,Net2,CNN(Fig1 4conv+2pool+fc256),Adam,50,0.029749,2.531165,54.271357,55.276382,44,1.600898,58.415842
3,Net2,CNN(Fig1 4conv+2pool+fc256),Adam,100,0.011616,3.146628,48.743719,55.276382,44,1.600898,58.415842
4,Net3,CNN+BN(Fig1 4conv+2pool+fc256),Adam,50,0.007314,1.388089,61.809045,65.326633,21,0.869634,73.267327
5,Net3,CNN+BN(Fig1 4conv+2pool+fc256),Adam,100,0.006064,1.404508,58.793970,65.326633,21,0.869634,73.267327
6,Net4,CNN+BN(Fig1 4conv+2pool+fc256),RMSprop,50,0.014717,1.393946,62.311558,62.311558,30,0.945145,70.297030
7,Net4,CNN+BN(Fig1 4conv+2pool+fc256),RMSprop,100,0.023469,1.782697,57.286432,62.311558,30,0.945145,70.297030


Saved results to results_image_models.csv


In [10]:
# Cell 18: Net5/Net6 audio constants

AUDIO_CLASS_NAMES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]
AUDIO_CLASS_TO_IDX = {name: idx for idx, name in enumerate(AUDIO_CLASS_NAMES)}

TARGET_SAMPLE_RATE = 22050
DURATION = 30
TARGET_NUM_SAMPLES = TARGET_SAMPLE_RATE * DURATION
N_FFT = 1024
HOP_LENGTH = 512
N_MELS = 128
N_MFCC = 40
AUDIO_FEATURE_DIM = N_MELS + (N_MFCC * 3)  # log-mel + MFCC + delta MFCC + delta-delta MFCC
MAX_AUDIO_TIME_STEPS = 256
AUDIO_BATCH_SIZE = 16
AUDIO_FULL_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 10

set_seed(SEED)
print("Audio pipeline configured. AUDIO_DIR:", AUDIO_DIR)
print("Audio feature dim:", AUDIO_FEATURE_DIM)


DEVICE: cpu
AUDIO_DIR: Data/genres_original
MAX_AUDIO_TIME_STEPS: 256
N_MELS: 128


In [11]:
# Cell 2: Collect and check readable audio files

def collect_audio_files(audio_dir):
    samples = []

    for class_name in AUDIO_CLASS_NAMES:
        class_dir = os.path.join(audio_dir, class_name)

        if not os.path.isdir(class_dir):
            warnings.warn(f"Missing class folder: {class_dir}")
            continue

        wav_paths = sorted(glob.glob(os.path.join(class_dir, "*.wav")))

        for wav_path in wav_paths:
            label = AUDIO_CLASS_TO_IDX[class_name]
            samples.append((wav_path, label))

    return samples


def check_readable_audio_files(samples):
    good_samples = []
    bad_samples = []

    for wav_path, label in samples:
        try:
            sr, audio_np = wavfile.read(wav_path)

            if audio_np is None or len(audio_np) == 0:
                bad_samples.append((wav_path, label, "empty audio"))
            else:
                good_samples.append((wav_path, label))

        except Exception as e:
            bad_samples.append((wav_path, label, str(e)))

    return good_samples, bad_samples


all_audio_samples_v2 = collect_audio_files(AUDIO_DIR)
good_audio_samples_v2, bad_audio_samples_v2 = check_readable_audio_files(all_audio_samples_v2)

print("Total discovered wav files:", len(all_audio_samples_v2))
print("Readable wav files:", len(good_audio_samples_v2))
print("Problematic wav files:", len(bad_audio_samples_v2))

if len(bad_audio_samples_v2) > 0:
    print("\nProblematic files:")
    for wav_path, label, error_msg in bad_audio_samples_v2:
        print(f"{wav_path} | label={label} | error={error_msg}")

Total discovered wav files: 1000
Readable wav files: 999
Problematic wav files: 1

Problematic files:
Data/genres_original\jazz\jazz.00054.wav | label=5 | error=File format b'\xcb\x15\x1e\x16' not understood. Only 'RIFF' and 'RIFX' supported.


In [12]:
# Cell 3: Mel filterbank helpers

def hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + hz / 700.0)


def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)


def create_mel_filterbank(sample_rate, n_fft, n_mels, f_min=0.0, f_max=None):
    if f_max is None:
        f_max = sample_rate / 2.0

    min_mel = hz_to_mel(f_min)
    max_mel = hz_to_mel(f_max)

    mel_points = np.linspace(min_mel, max_mel, n_mels + 2)
    hz_points = mel_to_hz(mel_points)

    fft_bins = np.floor((n_fft + 1) * hz_points / sample_rate).astype(int)

    filterbank = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float32)

    for m in range(1, n_mels + 1):
        left = fft_bins[m - 1]
        center = fft_bins[m]
        right = fft_bins[m + 1]

        if center == left:
            center += 1
        if right == center:
            right += 1

        for k in range(left, center):
            if 0 <= k < filterbank.shape[1]:
                filterbank[m - 1, k] = (k - left) / (center - left)

        for k in range(center, right):
            if 0 <= k < filterbank.shape[1]:
                filterbank[m - 1, k] = (right - k) / (right - center)

    return filterbank


MEL_FILTERBANK_V2 = create_mel_filterbank(
    sample_rate=TARGET_SAMPLE_RATE,
    n_fft=N_FFT,
    n_mels=N_MELS
)

print("Mel filterbank shape:", MEL_FILTERBANK_V2.shape)

Mel filterbank shape: (128, 513)


In [13]:
# Cell 4: Audio Feature Sequence Dataset V2

class AudioFeatureSequenceDatasetV2(Dataset):
    """
    Net5/Net6 audio dataset.

    Each audio file is converted into an audio feature sequence:
    waveform -> power spectrogram -> log-mel + MFCC + delta/delta-delta MFCC -> fixed time length

    Output shape:
    x: [MAX_AUDIO_TIME_STEPS, AUDIO_FEATURE_DIM]
    y: class label
    """

    def __init__(self, samples):
        self.samples = samples
        self.class_names = AUDIO_CLASS_NAMES
        self.class_to_idx = AUDIO_CLASS_TO_IDX
        self.feature_type = "log-mel + MFCC + delta + delta-delta"

        if len(self.samples) == 0:
            raise RuntimeError("No readable audio samples were provided.")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _to_float32_mono(audio_np):
        if audio_np.ndim > 1:
            audio_np = audio_np.mean(axis=1)

        if np.issubdtype(audio_np.dtype, np.integer):
            info = np.iinfo(audio_np.dtype)
            max_abs = max(abs(info.min), abs(info.max))
            audio_np = audio_np.astype(np.float32) / float(max_abs)
        else:
            audio_np = audio_np.astype(np.float32)

        return np.clip(audio_np, -1.0, 1.0)

    @staticmethod
    def _fit_waveform_length(waveform, target_len):
        if len(waveform) < target_len:
            waveform = np.pad(waveform, (0, target_len - len(waveform)), mode="constant")
        elif len(waveform) > target_len:
            waveform = waveform[:target_len]

        return waveform.astype(np.float32)

    @staticmethod
    def _fit_time_steps(features, target_steps):
        current_steps = features.shape[0]

        if current_steps == target_steps:
            return features.astype(np.float32)

        old_x = np.linspace(0, 1, current_steps)
        new_x = np.linspace(0, 1, target_steps)

        resized = np.zeros((target_steps, features.shape[1]), dtype=np.float32)

        for f in range(features.shape[1]):
            resized[:, f] = np.interp(new_x, old_x, features[:, f])

        return resized.astype(np.float32)

    def _extract_audio_feature_sequence(self, audio_np, sr):
        _, _, Sxx = spectrogram(
            audio_np,
            fs=sr,
            nperseg=N_FFT,
            noverlap=N_FFT - HOP_LENGTH,
            mode="magnitude"
        )

        power_spec = (Sxx ** 2).astype(np.float32)

        mel_spec = np.matmul(MEL_FILTERBANK_V2, power_spec).astype(np.float32)
        log_mel_spec = 10.0 * np.log10(mel_spec + 1e-10)
        log_mel_spec = np.maximum(log_mel_spec, log_mel_spec.max() - 80.0)

        mfcc = dct(log_mel_spec, type=2, axis=0, norm="ortho")[:N_MFCC, :].astype(np.float32)
        mfcc_delta = np.gradient(mfcc, axis=1).astype(np.float32)
        mfcc_delta2 = np.gradient(mfcc_delta, axis=1).astype(np.float32)

        feature_stack = np.concatenate([log_mel_spec, mfcc, mfcc_delta, mfcc_delta2], axis=0)
        feature_sequence = feature_stack.T.astype(np.float32)  # [time, feature_dim]
        feature_sequence = self._fit_time_steps(feature_sequence, MAX_AUDIO_TIME_STEPS)

        return feature_sequence

    def __getitem__(self, idx):
        wav_path, label = self.samples[idx]

        sr, audio_np = wavfile.read(wav_path)
        audio_np = self._to_float32_mono(audio_np)

        if sr != TARGET_SAMPLE_RATE:
            target_len = int(round(len(audio_np) * TARGET_SAMPLE_RATE / sr))
            audio_np = resample(audio_np, target_len).astype(np.float32)
            sr = TARGET_SAMPLE_RATE

        audio_np = self._fit_waveform_length(audio_np, TARGET_NUM_SAMPLES)

        features = self._extract_audio_feature_sequence(audio_np, sr)

        x = torch.tensor(features, dtype=torch.float32)
        y = torch.tensor(label, dtype=torch.long)

        return x, y


In [14]:
# Cell 22: Create audio dataset/dataloaders with train-set normalisation

set_seed(SEED)
audio_dataset_v2 = AudioFeatureSequenceDatasetV2(good_audio_samples_v2)

n_total = len(audio_dataset_v2)
n_train = int(0.7 * n_total)
n_val = int(0.2 * n_total)
n_test = n_total - n_train - n_val

split_generator = torch.Generator().manual_seed(SEED)
train_audio_dataset_v2, val_audio_dataset_v2, test_audio_dataset_v2 = random_split(
    audio_dataset_v2,
    [n_train, n_val, n_test],
    generator=split_generator,
)

print(f"Audio split sizes (70/20/10): train={len(train_audio_dataset_v2)}, val={len(val_audio_dataset_v2)}, test={len(test_audio_dataset_v2)}")

def audio_split_counts(subset):
    counts = {cls: 0 for cls in AUDIO_CLASS_NAMES}
    for idx in subset.indices:
        _, label = audio_dataset_v2.samples[idx]
        counts[AUDIO_CLASS_NAMES[label]] += 1
    return counts

print("Audio train class counts:", audio_split_counts(train_audio_dataset_v2))
print("Audio val class counts:", audio_split_counts(val_audio_dataset_v2))
print("Audio test class counts:", audio_split_counts(test_audio_dataset_v2))

def subset_to_tensors(subset):
    xs = []
    ys = []
    for i in range(len(subset)):
        x, y = subset[i]
        xs.append(x)
        ys.append(y)
    return torch.stack(xs), torch.tensor(ys, dtype=torch.long)

X_train_raw_v2, y_train_audio_v2 = subset_to_tensors(train_audio_dataset_v2)
X_val_raw_v2, y_val_audio_v2 = subset_to_tensors(val_audio_dataset_v2)
X_test_raw_v2, y_test_audio_v2 = subset_to_tensors(test_audio_dataset_v2)

# Train-set statistics only: mean/std over [batch, time], preserving feature_dim
train_feature_mean = X_train_raw_v2.mean(dim=(0, 1), keepdim=True)
train_feature_std = X_train_raw_v2.std(dim=(0, 1), keepdim=True).clamp_min(1e-6)

def apply_train_norm(x):
    return (x - train_feature_mean) / train_feature_std

X_train_audio_v2 = apply_train_norm(X_train_raw_v2)
X_val_audio_v2 = apply_train_norm(X_val_raw_v2)
X_test_audio_v2 = apply_train_norm(X_test_raw_v2)

train_audio_loader_v2 = DataLoader(TensorDataset(X_train_audio_v2, y_train_audio_v2), batch_size=AUDIO_BATCH_SIZE, shuffle=True, num_workers=0)
val_audio_loader_v2 = DataLoader(TensorDataset(X_val_audio_v2, y_val_audio_v2), batch_size=AUDIO_BATCH_SIZE, shuffle=False, num_workers=0)
test_audio_loader_v2 = DataLoader(TensorDataset(X_test_audio_v2, y_test_audio_v2), batch_size=AUDIO_BATCH_SIZE, shuffle=False, num_workers=0)

sample_x_v2, sample_y_v2 = next(iter(train_audio_loader_v2))
print("Net5 train feature shape:", tuple(X_train_audio_v2.shape))
print("Audio batch shape:", tuple(sample_x_v2.shape))
print("Audio label shape:", tuple(sample_y_v2.shape))
print("Audio value range:", float(sample_x_v2.min()), float(sample_x_v2.max()))


===== Audio preprocessing sanity check V2 =====
Feature type: dB log-mel spectrogram
Total usable samples: 999
Train / Val / Test: 699 199 101

Batch feature shape: torch.Size([16, 256, 128])
Batch label shape: torch.Size([16])

Whole batch feature statistics:
mean: 5.778565537184477e-08
std: 1.0000008344650269
min: -5.366174221038818
max: 5.708608627319336

Per-sample statistics for first 5 samples:
Sample 0: mean=0.0000, std=1.0000, min=-3.5710, max=2.8649, label=0
Sample 1: mean=0.0000, std=1.0000, min=-2.6420, max=2.9904, label=4
Sample 2: mean=0.0000, std=1.0000, min=-5.3662, max=4.0967, label=3
Sample 3: mean=-0.0000, std=1.0000, min=-3.5424, max=2.8381, label=6
Sample 4: mean=-0.0000, std=1.0000, min=-4.1109, max=3.3295, label=7

Labels:
labels in this batch: [0, 4, 3, 6, 7, 1, 4, 5, 9, 2, 7, 0, 0, 1, 4, 0]
min label: 0
max label: 9

Class counts:
blues: 100
classical: 100
country: 100
disco: 100
hiphop: 100
jazz: 99
metal: 100
pop: 100
reggae: 100
rock: 100


In [16]:
# Cell 6: Net5 LSTM V2

class Net5LSTMV2(nn.Module):
    """
    Net5: bidirectional LSTM for log-mel + MFCC-based audio feature sequences.

    Input shape: [batch, time_steps, feature_dim]
    Example: [16, 256, 248]
    """

    def __init__(
        self,
        input_size=AUDIO_FEATURE_DIM,
        hidden_size=128,
        num_layers=2,
        num_classes=NUM_CLASSES,
        dropout=0.3,
        bidirectional=True
    ):
        super(Net5LSTMV2, self).__init__()

        self.num_directions = 2 if bidirectional else 1

        self.input_norm = nn.LayerNorm(input_size)

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )

        lstm_output_dim = hidden_size * self.num_directions

        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.input_norm(x)

        outputs, _ = self.lstm(x)

        mean_pool = outputs.mean(dim=1)
        max_pool, _ = outputs.max(dim=1)

        pooled = torch.cat([mean_pool, max_pool], dim=1)

        logits = self.classifier(pooled)
        return logits


net5_v2_check = Net5LSTMV2().to(DEVICE)
sample_logits_v2 = net5_v2_check(sample_x_v2.to(DEVICE))

print("Sample input shape:", sample_x_v2.shape)
print("Sample output shape:", sample_logits_v2.shape)


Sample input shape: torch.Size([16, 256, 128])
Sample output shape: torch.Size([16, 10])


In [17]:
# Cell 7: Training and evaluation functions V2

def train_one_epoch_audio_v2(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in dataloader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item() * x.size(0)

        _, predicted = torch.max(outputs, dim=1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

    return running_loss / total, correct / total


def evaluate_audio_v2(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)
            loss = criterion(outputs, y)

            running_loss += loss.item() * x.size(0)

            _, predicted = torch.max(outputs, dim=1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

    return running_loss / total, correct / total


def train_audio_model_best_acc_v2(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs=AUDIO_FULL_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE
):
    best_val_acc = 0.0
    best_model_state = copy.deepcopy(model.state_dict())
    patience_counter = 0

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch_audio_v2(
            model, train_loader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate_audio_v2(
            model, val_loader, criterion, device
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch [{epoch + 1:02d}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    model.load_state_dict(best_model_state)

    return model, history


In [18]:
# Cell 25: Train Net5 (LSTM on log-mel + MFCC-based audio feature sequences)

if RUN_FULL_AUDIO_MODELS:
    print("\n===== Training Net5 (audio feature sequences, LSTM) =====")
    set_seed(SEED)
    net5_v2 = Net5LSTMV2().to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(net5_v2.parameters(), lr=5e-4, weight_decay=5e-4)

    net5_v2, net5_v2_history = train_audio_model_best_acc_v2(
        model=net5_v2,
        train_loader=train_audio_loader_v2,
        val_loader=val_audio_loader_v2,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        num_epochs=AUDIO_FULL_EPOCHS,
        patience=EARLY_STOPPING_PATIENCE,
    )

    net5_v2_test_loss, net5_v2_test_acc = evaluate_audio_v2(net5_v2, test_audio_loader_v2, criterion, DEVICE)
    best_net5_v2_val_acc = max(net5_v2_history["val_acc"])
    best_net5_v2_val_epoch = net5_v2_history["val_acc"].index(best_net5_v2_val_acc) + 1

    torch.save(net5_v2.state_dict(), os.path.join(CHECKPOINT_DIR, "net5_best.pt"))
    append_result(
        "Net5", "audio_logmel_mfcc_features", "BiLSTM(hidden128,layers2)", "AdamW", len(net5_v2_history["val_acc"]),
        {"val_acc": [v * 100.0 for v in net5_v2_history["val_acc"]]}, net5_v2_test_loss, net5_v2_test_acc * 100.0
    )
    save_results_csv(ALL_RESULTS_CSV)

    print(f"Net5 Test Loss: {net5_v2_test_loss:.4f}")
    print(f"Net5 Test Acc: {net5_v2_test_acc*100.0:.2f}%")
    print(f"Net5 Best Val Acc: {best_net5_v2_val_acc*100.0:.2f}% at epoch {best_net5_v2_val_epoch}")
else:
    print("Net5 training skipped (RUN_FULL_AUDIO_MODELS=False).")


Epoch [01/50] Train Loss: 2.2159 | Train Acc: 0.2089 Val Loss: 2.0638 | Val Acc: 0.3015
Epoch [02/50] Train Loss: 1.8920 | Train Acc: 0.3619 Val Loss: 1.8876 | Val Acc: 0.3518
Epoch [03/50] Train Loss: 1.7403 | Train Acc: 0.3777 Val Loss: 1.7045 | Val Acc: 0.4070
Epoch [04/50] Train Loss: 1.6079 | Train Acc: 0.4521 Val Loss: 1.7070 | Val Acc: 0.4422
Epoch [05/50] Train Loss: 1.5389 | Train Acc: 0.4692 Val Loss: 1.6200 | Val Acc: 0.4221
Epoch [06/50] Train Loss: 1.4256 | Train Acc: 0.5622 Val Loss: 1.6236 | Val Acc: 0.4523
Epoch [07/50] Train Loss: 1.3208 | Train Acc: 0.5551 Val Loss: 1.5295 | Val Acc: 0.5226
Epoch [08/50] Train Loss: 1.2537 | Train Acc: 0.5951 Val Loss: 1.4556 | Val Acc: 0.5075
Epoch [09/50] Train Loss: 1.1375 | Train Acc: 0.6552 Val Loss: 1.5388 | Val Acc: 0.5176
Epoch [10/50] Train Loss: 1.0924 | Train Acc: 0.6738 Val Loss: 1.5317 | Val Acc: 0.4673
Epoch [11/50] Train Loss: 1.0660 | Train Acc: 0.6867 Val Loss: 1.4447 | Val Acc: 0.5578
Epoch [12/50] Train Loss: 0.9580

In [24]:
# Net6 Step 1: Reuse normalised real Net5 training features
# Net6 performs GAN augmentation in the improved audio feature space,
# not by generating playable waveform audio.

X_train_real_v2 = X_train_audio_v2.clone()
y_train_real_v2 = y_train_audio_v2.clone()

print("Real train features:", X_train_real_v2.shape)
print("Real train labels:", y_train_real_v2.shape)
print("Feature mean:", X_train_real_v2.mean().item())
print("Feature std:", X_train_real_v2.std().item())
print("Label min/max:", y_train_real_v2.min().item(), y_train_real_v2.max().item())


Real train features: torch.Size([699, 256, 128])
Real train labels: torch.Size([699])
Feature mean: -1.0829456442706942e-08
Feature std: 0.9999999403953552
Label min/max: 0 9


In [25]:
# Net6 Step 2: Downsample real features for GAN

GAN_TIME_STEPS = 64
GAN_FEATURE_BINS = 64
GAN_FEATURE_SCALE = 5.0
GAN_BATCH_SIZE = 32
NOISE_DIM = 100
GAN_EPOCHS = 30

# [N, T, F] -> [N, 1, T, F]
X_train_gan = X_train_real_v2.unsqueeze(1)

# downsample to [N, 1, 64, 64]
X_train_gan = torch.nn.functional.interpolate(
    X_train_gan,
    size=(GAN_TIME_STEPS, GAN_FEATURE_BINS),
    mode="bilinear",
    align_corners=False
)

# scale roughly to [-1, 1] for tanh GAN output
X_train_gan = torch.clamp(X_train_gan / GAN_FEATURE_SCALE, -1.0, 1.0)

gan_train_dataset = TensorDataset(X_train_gan, y_train_real_v2)

gan_train_loader = DataLoader(
    gan_train_dataset,
    batch_size=GAN_BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

print("GAN train features:", X_train_gan.shape)
print("GAN feature min/max:", X_train_gan.min().item(), X_train_gan.max().item())


GAN train features: torch.Size([699, 1, 64, 64])
GAN feature min/max: -1.0 0.9768930673599243


In [26]:
# Net6 Step 3: Conditional GAN

class ConditionalGenerator(nn.Module):
    def __init__(self, noise_dim=NOISE_DIM, num_classes=NUM_CLASSES, label_dim=32):
        super().__init__()

        self.label_embedding = nn.Embedding(num_classes, label_dim)

        self.net = nn.Sequential(
            nn.Linear(noise_dim + label_dim, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),

            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),

            nn.Linear(512, GAN_TIME_STEPS * GAN_MEL_BINS),
            nn.Tanh()
        )

    def forward(self, z, labels):
        label_vec = self.label_embedding(labels)
        x = torch.cat([z, label_vec], dim=1)
        out = self.net(x)
        out = out.view(-1, 1, GAN_TIME_STEPS, GAN_MEL_BINS)
        return out


class ConditionalDiscriminator(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, label_dim=32):
        super().__init__()

        self.label_embedding = nn.Embedding(num_classes, label_dim)

        self.net = nn.Sequential(
            nn.Linear(GAN_TIME_STEPS * GAN_MEL_BINS + label_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            nn.Linear(256, 1)
        )

    def forward(self, x, labels):
        x = x.view(x.size(0), -1)
        label_vec = self.label_embedding(labels)
        x = torch.cat([x, label_vec], dim=1)
        logits = self.net(x)
        return logits

In [27]:
# Net6 Step 4: Train Conditional GAN

if RUN_FULL_AUDIO_MODELS and RUN_GAN:
    set_seed(SEED)
    generator = ConditionalGenerator().to(DEVICE)
    discriminator = ConditionalDiscriminator().to(DEVICE)

    gan_criterion = nn.BCEWithLogitsLoss()
    g_optimizer = optim.Adam(generator.parameters(), lr=1e-4, betas=(0.5, 0.999))
    d_optimizer = optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.5, 0.999))

    for epoch in range(GAN_EPOCHS):
        generator.train()
        discriminator.train()
        total_d_loss = 0.0
        total_g_loss = 0.0
        total_batches = 0

        for real_x, labels in gan_train_loader:
            real_x = real_x.to(DEVICE)
            labels = labels.to(DEVICE)
            batch_size = real_x.size(0)

            real_targets = torch.ones(batch_size, 1, device=DEVICE) * 0.9
            fake_targets = torch.zeros(batch_size, 1, device=DEVICE)

            d_optimizer.zero_grad()
            real_logits = discriminator(real_x, labels)
            d_real_loss = gan_criterion(real_logits, real_targets)

            z = torch.randn(batch_size, NOISE_DIM, device=DEVICE)
            fake_x = generator(z, labels)
            fake_logits = discriminator(fake_x.detach(), labels)
            d_fake_loss = gan_criterion(fake_logits, fake_targets)

            d_loss = d_real_loss + d_fake_loss
            d_loss.backward()
            d_optimizer.step()

            g_optimizer.zero_grad()
            z = torch.randn(batch_size, NOISE_DIM, device=DEVICE)
            gen_x = generator(z, labels)
            gen_logits = discriminator(gen_x, labels)
            g_loss = gan_criterion(gen_logits, real_targets)
            g_loss.backward()
            g_optimizer.step()

            total_d_loss += d_loss.item()
            total_g_loss += g_loss.item()
            total_batches += 1

        print(f"GAN Epoch [{epoch + 1:02d}/{GAN_EPOCHS}] D Loss: {total_d_loss/total_batches:.4f} | G Loss: {total_g_loss/total_batches:.4f}")
else:
    print("GAN training skipped (RUN_FULL_AUDIO_MODELS=False or RUN_GAN=False).")


GAN Epoch [01/30] D Loss: 0.5642 | G Loss: 0.8116
GAN Epoch [02/30] D Loss: 0.3542 | G Loss: 1.2013
GAN Epoch [03/30] D Loss: 0.2655 | G Loss: 1.7866
GAN Epoch [04/30] D Loss: 0.2580 | G Loss: 2.1703
GAN Epoch [05/30] D Loss: 0.3269 | G Loss: 2.2357
GAN Epoch [06/30] D Loss: 0.3769 | G Loss: 2.3231
GAN Epoch [07/30] D Loss: 0.3605 | G Loss: 2.3285
GAN Epoch [08/30] D Loss: 0.3573 | G Loss: 2.3607
GAN Epoch [09/30] D Loss: 0.3278 | G Loss: 2.5783
GAN Epoch [10/30] D Loss: 0.3051 | G Loss: 2.7536
GAN Epoch [11/30] D Loss: 0.2866 | G Loss: 3.0560
GAN Epoch [12/30] D Loss: 0.2795 | G Loss: 3.3315
GAN Epoch [13/30] D Loss: 0.2838 | G Loss: 3.7010
GAN Epoch [14/30] D Loss: 0.2934 | G Loss: 4.0660
GAN Epoch [15/30] D Loss: 0.3174 | G Loss: 4.2634
GAN Epoch [16/30] D Loss: 0.3175 | G Loss: 4.2568
GAN Epoch [17/30] D Loss: 0.3087 | G Loss: 4.2742
GAN Epoch [18/30] D Loss: 0.3190 | G Loss: 4.3830
GAN Epoch [19/30] D Loss: 0.3032 | G Loss: 3.8890
GAN Epoch [20/30] D Loss: 0.2869 | G Loss: 4.3132


In [28]:
# Net6 Step 5: Generate synthetic features

def apply_train_norm_for_generated(x):
    # x shape: [N, MAX_AUDIO_TIME_STEPS, AUDIO_FEATURE_DIM]
    return (x - train_feature_mean) / train_feature_std


generator.eval()

synthetic_features = []
synthetic_labels = []

num_synthetic = len(X_train_real_v2)
batch_size = 64

with torch.no_grad():
    for start in range(0, num_synthetic, batch_size):
        end = min(start + batch_size, num_synthetic)
        current_batch_size = end - start

        # sample labels from real train labels to preserve class distribution
        label_indices = torch.randint(0, len(y_train_real_v2), (current_batch_size,))
        labels = y_train_real_v2[label_indices].to(DEVICE)

        z = torch.randn(current_batch_size, NOISE_DIM, device=DEVICE)

        fake_lowres = generator(z, labels)

        # [B, 1, 64, 64] -> [B, 1, 256, AUDIO_FEATURE_DIM]
        fake_highres = torch.nn.functional.interpolate(
            fake_lowres,
            size=(MAX_AUDIO_TIME_STEPS, AUDIO_FEATURE_DIM),
            mode="bilinear",
            align_corners=False
        )

        fake_highres = fake_highres.squeeze(1).cpu()

        # rescale from [-1,1] roughly back to feature scale, then apply train-set normalisation
        fake_highres = fake_highres * GAN_FEATURE_SCALE
        fake_highres = apply_train_norm_for_generated(fake_highres)

        synthetic_features.append(fake_highres)
        synthetic_labels.append(labels.cpu())

X_synthetic_v2 = torch.cat(synthetic_features, dim=0)
y_synthetic_v2 = torch.cat(synthetic_labels, dim=0)

print("Synthetic features:", X_synthetic_v2.shape)
print("Synthetic labels:", y_synthetic_v2.shape)
print("Synthetic mean:", X_synthetic_v2.mean().item())
print("Synthetic std:", X_synthetic_v2.std().item())
print("Synthetic min/max:", X_synthetic_v2.min().item(), X_synthetic_v2.max().item())


Synthetic features: torch.Size([699, 256, 128])
Synthetic labels: torch.Size([699])
Synthetic mean: -1.0872092115477017e-09
Synthetic std: 0.9999838471412659
Synthetic min/max: -5.087758541107178 4.992560863494873


In [29]:
# Net6 Step 6: Create augmented training loader

X_train_net6 = torch.cat([X_train_real_v2, X_synthetic_v2], dim=0)
y_train_net6 = torch.cat([y_train_real_v2, y_synthetic_v2], dim=0)

net6_train_dataset = TensorDataset(X_train_net6, y_train_net6)

net6_train_loader = DataLoader(
    net6_train_dataset,
    batch_size=AUDIO_BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

print("Net6 augmented train features:", X_train_net6.shape)
print("Net6 augmented train labels:", y_train_net6.shape)

Net6 augmented train features: torch.Size([1398, 256, 128])
Net6 augmented train labels: torch.Size([1398])


In [30]:
# Net6 Step 7: Train Net6 classifier using augmented data

if RUN_FULL_AUDIO_MODELS and RUN_GAN:
    print("\n===== Training Net6 (Net5 architecture + GAN-augmented audio feature sequences) =====")
    set_seed(SEED)
    net6_v2 = Net5LSTMV2().to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(net6_v2.parameters(), lr=5e-4, weight_decay=5e-4)

    net6_v2, net6_v2_history = train_audio_model_best_acc_v2(
        model=net6_v2,
        train_loader=net6_train_loader,
        val_loader=val_audio_loader_v2,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        num_epochs=AUDIO_FULL_EPOCHS,
        patience=EARLY_STOPPING_PATIENCE,
    )

    net6_v2_test_loss, net6_v2_test_acc = evaluate_audio_v2(net6_v2, test_audio_loader_v2, criterion, DEVICE)
    best_net6_v2_val_acc = max(net6_v2_history["val_acc"])
    best_net6_v2_val_epoch = net6_v2_history["val_acc"].index(best_net6_v2_val_acc) + 1

    torch.save(net6_v2.state_dict(), os.path.join(CHECKPOINT_DIR, "net6_best.pt"))
    append_result(
        "Net6", "audio_logmel_mfcc_features_gan_aug", "BiLSTM(hidden128,layers2)", "AdamW", len(net6_v2_history["val_acc"]),
        {"val_acc": [v * 100.0 for v in net6_v2_history["val_acc"]]}, net6_v2_test_loss, net6_v2_test_acc * 100.0
    )
    save_results_csv(ALL_RESULTS_CSV)

    print(f"Net6 Test Loss: {net6_v2_test_loss:.4f}")
    print(f"Net6 Test Acc: {net6_v2_test_acc*100.0:.2f}%")
    print(f"Net6 Best Val Acc: {best_net6_v2_val_acc*100.0:.2f}% at epoch {best_net6_v2_val_epoch}")
else:
    print("Net6 training skipped (RUN_FULL_AUDIO_MODELS=False or RUN_GAN=False).")


Epoch [01/50] Train Loss: 2.1010 | Train Acc: 0.2918 Val Loss: 2.0105 | Val Acc: 0.2965
Epoch [02/50] Train Loss: 1.6319 | Train Acc: 0.4735 Val Loss: 1.8054 | Val Acc: 0.4121
Epoch [03/50] Train Loss: 1.3628 | Train Acc: 0.5794 Val Loss: 1.6959 | Val Acc: 0.4221
Epoch [04/50] Train Loss: 1.1797 | Train Acc: 0.6567 Val Loss: 1.5993 | Val Acc: 0.4824
Epoch [05/50] Train Loss: 1.0457 | Train Acc: 0.7060 Val Loss: 1.5823 | Val Acc: 0.4874
Epoch [06/50] Train Loss: 0.9228 | Train Acc: 0.7539 Val Loss: 1.5198 | Val Acc: 0.4925
Epoch [07/50] Train Loss: 0.8374 | Train Acc: 0.7868 Val Loss: 1.5745 | Val Acc: 0.4975
Epoch [08/50] Train Loss: 0.8060 | Train Acc: 0.8004 Val Loss: 1.5809 | Val Acc: 0.5477
Epoch [09/50] Train Loss: 0.7117 | Train Acc: 0.8462 Val Loss: 1.4816 | Val Acc: 0.5276
Epoch [10/50] Train Loss: 0.6648 | Train Acc: 0.8612 Val Loss: 1.5320 | Val Acc: 0.5427
Epoch [11/50] Train Loss: 0.6371 | Train Acc: 0.8705 Val Loss: 1.4561 | Val Acc: 0.5578
Epoch [12/50] Train Loss: 0.6071

In [ ]:
# Cell 33: Final combined summary and consistency checks

final_df = save_results_csv(ALL_RESULTS_CSV)
print("\n===== Final Summary Table =====")
display(final_df)

required = {
    ("Net1", 50), ("Net1", 100),
    ("Net2", 50), ("Net2", 100),
    ("Net3", 50), ("Net3", 100),
    ("Net4", 50), ("Net4", 100),
}
existing = {(r["model_name"], int(r["epochs_run"])) for _, r in final_df.iterrows() if r["model_name"] in {"Net1","Net2","Net3","Net4"}}
missing = sorted(required - existing)

print("Consistency check:")
print("- Net1-Net4 are image models:", set(final_df[final_df["model_name"].isin(["Net1","Net2","Net3","Net4"])] ["input_type"].tolist()))
print("- Net5-Net6 are audio feature-sequence models:", set(final_df[final_df["model_name"].isin(["Net5","Net6"])] ["input_type"].tolist()))
print("- Missing Net1-Net4 (50/100) runs:", missing if missing else "None")
print("- Net3 defined as Net2 + BatchNorm: see model definitions above")
print("- Net4 defined as Net3 + RMSprop: see Net4 setup cell")
print("- Net5 uses log-mel + MFCC-based audio feature sequences with BiLSTM + mean/max pooling")
print("- Net6 uses the same LSTM and performs GAN augmentation in audio feature space (not raw waveform generation)")
print(f"- Combined results saved to {ALL_RESULTS_CSV}")
